# Qualidade e preparação dos dados

## Objetivo

Avaliar a qualidade dos dados fornecidos, identificar inconsistências e preparar uma base para tratamento.

## Estrutura do notebook

1. Leitura dos dados da camada Bronze
2. Análise inicial das bases
3. Identificação de inconsistências
4. Resumo das inconsistências identificadas

In [1]:
import pandas as pd

ativos = pd.read_csv('../data/bronze/ativos.csv', sep=';')
vulnerabilidades = pd.read_csv('../data/bronze/vulnerabilidades.csv', sep=';')


## 1. Análise inicial das bases

In [2]:
ativos.head()

,ativo_id,sistema,criticidade,dominio,ambiente,localidade,status_ativo
0,A0001,Sistema_1,Crítica,Dominio_2,Homologação,São Paulo,Em desativação
1,A0002,Sistema_2,Média,Dominio_3,Produção,São Paulo,Em desativação
2,A0003,Sistema_3,Média,Dominio_4,Homologação,São Paulo,Ativo
3,A0004,Sistema_4,Alta,Dominio_5,Homologação,Campinas,Ativo
4,A0005,Sistema_5,Alta,Dominio_6,Produção,Recife,Ativo


In [3]:
ativos.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   ativo_id      200 non-null    str  
 1   sistema       200 non-null    str  
 2   criticidade   199 non-null    str  
 3   dominio       200 non-null    str  
 4   ambiente      198 non-null    str  
 5   localidade    200 non-null    str  
 6   status_ativo  200 non-null    str  
dtypes: str(7)
memory usage: 11.1 KB


In [4]:
vulnerabilidades.head()

,vuln_id,ativo_id,severidade,origem,data_abertura,data_correcao,status,cvss_score
0,V00001,A0078,Baixa,Cloud Security,21/03/2026,NaN,Em tratamento,2.8
1,V00002,A0060,Alta,Qualys,01/06/2026,NaN,Em tratamento,7.5
2,V00003,A0080,Alta,Veracode,18/08/2026,NaN,Aberta,1.7
3,V00004,A0086,Alta,Qualys,16/06/2026,NaN,Aberta,2.5
4,V00005,A0089,Alta,Cloud Security,12/02/2026,NaN,Em tratamento,4.4


In [5]:
vulnerabilidades.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   vuln_id        2000 non-null   str    
 1   ativo_id       2000 non-null   str    
 2   severidade     1974 non-null   str    
 3   origem         2000 non-null   str    
 4   data_abertura  2000 non-null   str    
 5   data_correcao  469 non-null    str    
 6   status         2000 non-null   str    
 7   cvss_score     2000 non-null   float64
dtypes: float64(1), str(7)
memory usage: 125.1 KB


## 2. Identificação de inconsistências

### Valores ausentes

In [6]:
print(f'Valores ausentes em ativos: Total {ativos.isna().sum().sum()}')
display(ativos.isna().sum())

print('-' * 50)

print(f'Valores ausentes em vulnerabilidades: Total {vulnerabilidades.isna().sum().sum()}')
display(vulnerabilidades.isna().sum())


Valores ausentes em ativos: Total 3


ativo_id        0
sistema         0
criticidade     1
dominio         0
ambiente        2
localidade      0
status_ativo    0
dtype: int64

--------------------------------------------------
Valores ausentes em vulnerabilidades: Total 1557


vuln_id             0
ativo_id            0
severidade         26
origem              0
data_abertura       0
data_correcao    1531
status              0
cvss_score          0
dtype: int64

### Validação entre status e data de correção

In [7]:
vulnerabilidades_status_corrigida = vulnerabilidades[vulnerabilidades['status'] == 'Corrigida']

vulnerabilidades_status_corrigida_na = vulnerabilidades_status_corrigida[vulnerabilidades_status_corrigida['data_correcao'].isna()]
vulnerabilidades_status_corrigida_na

,vuln_id,ativo_id,severidade,origem,data_abertura,data_correcao,status,cvss_score


In [8]:
vulnerabilidades_status_nao_corrigida = vulnerabilidades[vulnerabilidades['status'] != 'Corrigida']

vulnerabilidades_status_nao_corrigida_not_na = vulnerabilidades_status_nao_corrigida[vulnerabilidades_status_nao_corrigida['data_correcao'].notna()]
vulnerabilidades_status_nao_corrigida_not_na

,vuln_id,ativo_id,severidade,origem,data_abertura,data_correcao,status,cvss_score


**Conclusão:** Não foram encontradas vulnerabilidades corrigidas sem data de correção, nem vulnerabilidades não corrigidas com data de correção. Portanto, a ausência de `data_correcao` é esperada para os demais status.

### Validação do formato de datas

In [9]:
data_abertura_validada = pd.to_datetime(vulnerabilidades['data_abertura'], format='%d/%m/%Y', errors='coerce')
data_abertura_invalida = vulnerabilidades[data_abertura_validada.isna()]
print(f'Total de datas de abertura inválidas: {data_abertura_invalida.shape[0]}')
data_abertura_invalida.head()

Total de datas de abertura inválidas: 78


,vuln_id,ativo_id,severidade,origem,data_abertura,data_correcao,status,cvss_score
46,V00047,A0167,Alta,Pentest,03-29-2026,NaN,Em tratamento,2.6
70,V00071,A0200,Alta,Qualys,2026-13-01,31/08/2026,Corrigida,4.8
88,V00089,A0123,Média,Cloud Security,31/02/2026,NaN,Aceita,1.0
93,V00094,A0150,Baixa,Veracode,06-16-2026,NaN,Aberta,3.8
140,V00141,A0138,Crítica,Cloud Security,05-30-2026,17/06/2026,Corrigida,2.8


Foram identificadas 78 datas de abertura inválidas. Há registros em formatos diferentes do esperado, como `mm-dd-aaaa` e `aaaa-dd-mm`, além de datas inexistentes, como `31/02/2026`.

In [10]:
data_correcao_notna = vulnerabilidades[vulnerabilidades['data_correcao'].notna()]
data_correcao_validada = pd.to_datetime(data_correcao_notna['data_correcao'], format='%d/%m/%Y', errors='coerce')
data_correcao_invalida = data_correcao_notna[data_correcao_validada.isna()]
print(f'Total de datas de correção inválidas: {data_correcao_invalida.shape[0]}')
data_correcao_invalida.head()


Total de datas de correção inválidas: 0


,vuln_id,ativo_id,severidade,origem,data_abertura,data_correcao,status,cvss_score


Não foram identificadas datas de correção inválidas entre os registros preenchidos.

### Coerência temporal das datas

In [11]:
datas_incoerentes = data_correcao_notna[data_correcao_validada < data_abertura_validada.loc[data_correcao_notna.index]]
datas_incoerentes

,vuln_id,ativo_id,severidade,origem,data_abertura,data_correcao,status,cvss_score
1879,V01880,A0027,Alta,Cloud Security,09/12/2026,13/09/2026,Corrigida,8.4
1926,V01927,A0061,Crítica,Veracode,01/05/2026,07/03/2026,Corrigida,7.2


Foram identificadas 2 inconsistências com `data_correcao` anterior à `data_abertura`.

### Consistência de valores categóricos

In [12]:
vulnerabilidades['severidade'].value_counts(dropna=False)


severidade
Média      489
Crítica    487
Alta       472
Baixa      463
NaN         26
media       16
BaixA       16
ALTA        15
critica     13
Critica      1
Media        1
Alto         1
Name: count, dtype: int64

In [13]:
vulnerabilidades['origem'].value_counts(dropna=False)

origem
Veracode          522
Pentest           507
Cloud Security    504
Qualys            465
CloudSecurity       1
Cloud-Security      1
Name: count, dtype: int64

In [14]:
vulnerabilidades['status'].value_counts(dropna=False)

status
Em tratamento    526
Aceita           517
Aberta           488
Corrigida        469
Name: count, dtype: int64

In [15]:
ativos['criticidade'].value_counts(dropna=False)

criticidade
Baixa      56
Média      46
Crítica    45
Alta       42
media       4
            3
BaixA       2
critica     1
NaN         1
Name: count, dtype: int64

In [16]:
ativos['ambiente'].value_counts(dropna=False)

ambiente
Produção           75
Homologação        65
Desenvolvimento    58
NaN                 2
Name: count, dtype: int64

In [17]:
ativos['localidade'].value_counts(dropna=False)

localidade
Recife       53
Campinas     50
São Paulo    49
Curitiba     48
Name: count, dtype: int64

In [18]:
ativos['status_ativo'].value_counts(dropna=False)

status_ativo
Ativo             151
Em desativação     49
Name: count, dtype: int64

### Integridade dos identificadores

In [19]:
ativos['ativo_id'].duplicated().sum()

np.int64(0)

In [20]:
vulnerabilidades['vuln_id'].duplicated().sum()

np.int64(0)

In [21]:
ativo_id_verificados = vulnerabilidades['ativo_id'].isin(ativos['ativo_id']).value_counts()
print (ativo_id_verificados)
print('-' * 50)
print('Total de vulnerabilidades com ativo_id não presente em ativos:')
display(vulnerabilidades[vulnerabilidades['ativo_id'].isin(ativos['ativo_id']) == False])

ativo_id
True     1995
False       5
Name: count, dtype: int64
--------------------------------------------------
Total de vulnerabilidades com ativo_id não presente em ativos:


,vuln_id,ativo_id,severidade,origem,data_abertura,data_correcao,status,cvss_score
100,V00101,A9999,Média,Cloud Security,08/06/2026,NaN,Aceita,3.9
500,V00501,A9999,Crítica,Veracode,07/05/2026,NaN,Em tratamento,4.7
900,V00901,A9999,Baixa,Cloud Security,04/01/2026,29/03/2026,Corrigida,5.8
1300,V01301,A9999,Baixa,Cloud Security,21/07/2026,23/09/2026,Corrigida,7.5
1700,V01701,A9999,Crítica,Cloud Security,08/04/2026,31/05/2026,Corrigida,5.8


Não foram identificados valores duplicados nas chaves primárias `ativo_id` da base de ativos e `vuln_id` da base de vulnerabilidades.

Foram identificadas 5 vulnerabilidades associadas ao ativo `A9999`, que não existe na base de ativos. Esse caso representa uma quebra de integridade referencial e deverá ser tratado na camada Silver.

### Validação do intervalo de CVSS

In [22]:
vulnerabilidades[(vulnerabilidades['cvss_score'] >10) | (vulnerabilidades['cvss_score'] <0)]

,vuln_id,ativo_id,severidade,origem,data_abertura,data_correcao,status,cvss_score


Não foram identificados valores de `cvss_score` fora do intervalo esperado de 0 a 10.

## 3. Resumo das inconsistências identificadas

- Foram identificados 3 valores nulos na base de ativos: 1 em `criticidade` e 2 em `ambiente`. Além disso, foram encontrados 3 valores em branco em `criticidade`, que também deverão ser tratados como ausentes.
- Foram identificados 26 valores ausentes em `severidade`.
- A ausência de `data_correcao` é esperada para vulnerabilidades que não estão corrigidas.
- Foram identificadas 78 datas de abertura inválidas.
- Não foram identificadas datas de correção inválidas.
- Foram identificadas 2 vulnerabilidades com data de correção anterior à data de abertura.
- Foram encontradas variações de preenchimento em `severidade`, `criticidade` e `origem`.
- Foram identificadas 5 vulnerabilidades associadas ao ativo inexistente `A9999`.
- Não foram encontrados IDs duplicados nem valores de CVSS fora do intervalo de 0 a 10.
- As colunas `data_abertura` e `data_correcao` estão como texto (`str`), mas precisam ser convertidas para o tipo data (`datetime`).